### Data Ingestion for BNSS 2023

In [1]:
import os
import re
import fitz
import numpy as np

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from pymongo import MongoClient, ReplaceOne

c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
PDF_PATH = r"./THE_BHARATIYA_NAGARIK_SURAKSHA_SANHITA_2023.pdf"
LAW = "BNSS"
MAX_SECTION = 531
VECTOR_INDEX_NAME = "test_bns"
DB_NAME = "test_db"
COLLECTION_NAME = "test_collection"

In [3]:
def extract_main_text(page):

    blocks = page.get_text("blocks")

    kept_blocks = []

    for block in blocks:

        x0, y0, x1, y1, text, *_ = block

        # Main central statute column
        if (
            x0 >= 110
            and x1 <= 490
            and 80 <= y0 <= 780
        ):

            text = text.strip()

            if not text:
                continue

            upper = text.upper()

            # Remove Gazette headers
            if "THE GAZETTE OF INDIA" in upper:
                continue

            if "[PART II" in upper:
                continue

            kept_blocks.append(
                (y0, x0, text)
            )

    # maintain reading order
    kept_blocks.sort(
        key=lambda x: (x[0], x[1])
    )

    return "\n".join(
        text
        for _, _, text in kept_blocks
    )

In [4]:
pdf = fitz.open(PDF_PATH)

full_text = ""

for page_number, page in enumerate(
    pdf,
    start=1
):

    page_text = extract_main_text(
        page
    )

    full_text += (
        f"\n<<<PAGE:{page_number}>>>\n"
        + page_text
        + "\n"
    )


print("PDF pages:", len(pdf))

print(
    "Extracted characters:",
    len(full_text)
)

PDF pages: 249
Extracted characters: 643344


In [5]:
schedule_position = full_text.find(
    "THE FIRST SCHEDULE"
)

print(
    "First Schedule position:",
    schedule_position
)

First Schedule position: 543909


In [6]:
if schedule_position != -1:

    statute_text = full_text[
        :schedule_position
    ]

else:

    statute_text = full_text

In [7]:
print(
    "Statute characters:",
    len(statute_text)
)

Statute characters: 543909


In [8]:
def parse_bnss_sections(
    text,
    pdf_path
):

    starts = []

    search_position = 0

    for section_number in range(
        1,
        MAX_SECTION + 1
    ):

        pattern = re.compile(
            rf"(?m)^\s*{section_number}\.\s*"
        )

        match = pattern.search(
            text,
            search_position
        )

        if match is None:

            raise ValueError(
                f"BNSS Section "
                f"{section_number} not found"
            )

        starts.append(
            (
                section_number,
                match
            )
        )

        search_position = (
            match.end()
        )

    sections = []

    for i, (
        section_number,
        match
    ) in enumerate(starts):

        start = match.start()

        if i + 1 < len(starts):

            end = starts[
                i + 1
            ][1].start()

        else:

            end = len(text)

        raw_section = text[
            start:end
        ]

        # identify pages section came from
        pages = [
            int(page)
            for page in re.findall(
                r"<<<PAGE:(\d+)>>>",
                raw_section
            )
        ]

        # remove our internal markers
        clean_text = re.sub(
            r"<<<PAGE:\d+>>>",
            "",
            raw_section
        ).strip()

        sections.append(

            Document(

                page_content=
                    clean_text,

                metadata={
                    "law":
                        "BNSS",

                    "section":
                        str(section_number),

                    "pages":
                        sorted(
                            set(pages)
                        ),

                    "source":
                        pdf_path,

                    "document_type":
                        "primary_statute"
                }
            )
        )

    return sections

In [9]:
sections = parse_bnss_sections(
    statute_text,
    PDF_PATH
)

print(
    "Sections extracted:",
    len(sections)
)

Sections extracted: 531


In [10]:
actual_sections = [

    int(
        doc.metadata["section"]
    )

    for doc in sections
]

expected_sections = list(
    range(1, 532)
)


print(
    "All sections valid:",
    actual_sections
    == expected_sections
)

print(
    "First 10:",
    actual_sections[:10]
)

print(
    "Last 10:",
    actual_sections[-10:]
)

All sections valid: True
First 10: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Last 10: [522, 523, 524, 525, 526, 527, 528, 529, 530, 531]


In [11]:
assert (
    actual_sections
    == expected_sections
)

In [12]:
def show_section(number):

    for doc in sections:

        if (
            doc.metadata["section"]
            == str(number)
        ):

            print(
                "LAW:",
                doc.metadata["law"]
            )

            print(
                "SECTION:",
                doc.metadata["section"]
            )

            print(
                "PAGES:",
                doc.metadata["pages"]
            )

            print("\nTEXT:\n")

            print(
                doc.page_content
            )

            return

In [13]:
show_section(35)

LAW: BNSS
SECTION: 35
PAGES: [14]

TEXT:

35. (1) Any police officer may without an order from a Magistrate and without a
warrant, arrest any person—
(a) who commits, in the presence of a police officer, a cognizable offence; or
(b) against whom a reasonable complaint has been made, or credible information
has been received, or a reasonable suspicion exists that he has committed a cognizable
offence punishable with imprisonment for a term which may be less than seven years
or which may extend to seven years whether with or without fine, if the following
conditions are satisfied, namely:—
(i) the police officer has reason to believe on the basis of such complaint,
information, or suspicion that such person has committed the said offence;
(ii) the police officer is satisfied that such arrest is necessary—
(a) to prevent such person from committing any further offence; or
(b) for proper investigation of the offence; or
(c) to prevent such person from causing the evidence of the offence
to

In [14]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    "Embedding dimension:",
    model.get_sentence_embedding_dimension()
)

print(
    "Model max sequence:",
    model.max_seq_length
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2387.61it/s]


Embedding dimension: 384
Model max sequence: 256


C:\Users\Himanshu\AppData\Local\Temp\ipykernel_13416\494620326.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


In [15]:
splitter = RecursiveCharacterTextSplitter(

    chunk_size=650,

    chunk_overlap=80,

    separators=[
        "\n\n",
        "\nExplanation",
        "\nIllustration",
        "\nIllustrations",
        "\nProvided",
        "\n(",
        ". ",
        " "
    ]
)

In [16]:
def token_count(text):

    return len(
        model.tokenizer.tokenize(
            text
        )
    )

In [17]:
def create_chunks(
    sections
):

    final_chunks = []

    for section in sections:

        section_text = (
            section.page_content
        )

        # ------------------
        # Short section
        # ------------------

        if (
            token_count(section_text)
            <= 180
        ):

            parts = [
                section_text
            ]

        # ------------------
        # Long section
        # ------------------

        else:

            parts = (
                splitter.split_text(
                    section_text
                )
            )

        safe_parts = []

        # Additional MiniLM safety check
        for part in parts:

            if (
                token_count(part)
                <= 210
            ):

                safe_parts.append(
                    part
                )

            else:

                smaller_splitter = (
                    RecursiveCharacterTextSplitter(

                        chunk_size=400,

                        chunk_overlap=50,

                        separators=[
                            "\n\n",
                            ". ",
                            " "
                        ]
                    )
                )

                safe_parts.extend(
                    smaller_splitter
                    .split_text(part)
                )

        # Convert to Documents
        for index, part in enumerate(
            safe_parts
        ):

            final_chunks.append(

                Document(

                    page_content=part,

                    metadata={
                        **section.metadata,

                        "parent_section":
                            section.metadata[
                                "section"
                            ],

                        "chunk_index":
                            index
                    }
                )
            )

    return final_chunks

In [18]:
chunks = create_chunks(
    sections
)

print(
    "Parent sections:",
    len(sections)
)

print(
    "Child chunks:",
    len(chunks)
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1257 > 256). Running this sequence through the model will result in indexing errors


Parent sections: 531
Child chunks: 1279


In [19]:
def build_embedding_text(
    chunk
):

    return f"""
Bharatiya Nagarik Suraksha Sanhita, 2023
BNSS Section {chunk.metadata['section']}

{chunk.page_content}
""".strip()

In [20]:
invalid_chunks = []

max_tokens = 0

largest_chunk = None


for chunk in chunks:

    text = build_embedding_text(
        chunk
    )

    count = token_count(
        text
    )

    if count > max_tokens:

        max_tokens = count

        largest_chunk = (
            chunk
        )

    if (
        count
        > model.max_seq_length
    ):

        invalid_chunks.append({

            "section":
                chunk.metadata[
                    "section"
                ],

            "chunk":
                chunk.metadata[
                    "chunk_index"
                ],

            "tokens":
                count
        })


print(
    "Maximum embedding tokens:",
    max_tokens
)

print(
    "Model limit:",
    model.max_seq_length
)

print(
    "Invalid chunks:",
    len(invalid_chunks)
)

print(
    invalid_chunks[:10]
)

Maximum embedding tokens: 195
Model limit: 256
Invalid chunks: 0
[]


In [21]:
assert (
    len(invalid_chunks)
    == 0
)

In [22]:
embedding_texts = [

    build_embedding_text(
        chunk
    )

    for chunk in chunks
]

In [23]:
embeddings = model.encode(

    embedding_texts,

    batch_size=32,

    normalize_embeddings=True,

    show_progress_bar=True
)

Batches: 100%|██████████| 40/40 [00:29<00:00,  1.34it/s]


In [24]:
print(
    "Chunks:",
    len(chunks)
)

print(
    "Embeddings:",
    len(embeddings)
)

print(
    "Shape:",
    embeddings.shape
)

Chunks: 1279
Embeddings: 1279
Shape: (1279, 384)


In [29]:
MONGO_URI = os.getenv(
    "MONGODB_URI"
)

client = MongoClient(
    MONGO_URI
)

db = client[
    DB_NAME
]

collection = db[
    COLLECTION_NAME
]


print(
    client.admin.command(
        "ping"
    )
)

{'ok': 1}


In [30]:
indexes = list(
    collection.list_search_indexes()
)

for index in indexes:

    print(
        "Name:",
        index.get("name")
    )

    print(
        "Status:",
        index.get("status")
    )

    print(
        "Queryable:",
        index.get("queryable")
    )

    print("-" * 50)

Name: test_bns
Status: READY
Queryable: True
--------------------------------------------------


In [31]:
mongo_documents = []


for i, chunk in enumerate(
    chunks
):

    section = (
        chunk.metadata[
            "section"
        ]
    )

    chunk_index = (
        chunk.metadata[
            "chunk_index"
        ]
    )


    mongo_documents.append({

        "id":
            f"bnss-{section}-{chunk_index}",

        "values":
            embeddings[i].tolist(),

        "metadata": {

            "law":
                "BNSS",

            "section":
                section,

            "parent_section":
                section,

            "chunk_index":
                chunk_index,

            "document_type":
                "primary_statute",

            "text":
                chunk.page_content,

            "pages":
                chunk.metadata[
                    "pages"
                ],

            "source":
                os.path.basename(
                    PDF_PATH
                )
        }
    })


print(
    "Prepared:",
    len(mongo_documents)
)

Prepared: 1279


In [32]:
operations = [

    ReplaceOne(
        {
            "id":
                doc["id"]
        },

        doc,

        upsert=True
    )

    for doc in mongo_documents
]

In [33]:
result = collection.bulk_write(
    operations
)


print(
    "BNSS records in MongoDB:",
    collection.count_documents(
        {
            "metadata.law":
                "BNSS"
        }
    )
)

BNSS records in MongoDB: 1279


In [34]:
section_35 = list(
    collection.find(
        {
            "metadata.law":
                "BNSS",

            "metadata.section":
                "35"
        },

        {
            "_id": 0
        }
    )
)


print(
    "Section 35 chunks:",
    len(section_35)
)


for doc in section_35:

    print(
        "\n",
        doc["metadata"]["text"]
    )

Section 35 chunks: 11

 35. (1) Any police officer may without an order from a Magistrate and without a
warrant, arrest any person—
(a) who commits, in the presence of a police officer, a cognizable offence; or
(b) against whom a reasonable complaint has been made, or credible information
has been received, or a reasonable suspicion exists that he has committed a cognizable
offence punishable with imprisonment for a term which may be less than seven years
or which may extend to seven years whether with or without fine, if the following
conditions are satisfied, namely:—

 (i) the police officer has reason to believe on the basis of such complaint,
information, or suspicion that such person has committed the said offence;
(ii) the police officer is satisfied that such arrest is necessary—
(a) to prevent such person from committing any further offence; or
(b) for proper investigation of the offence; or
(c) to prevent such person from causing the evidence of the offence
to disappear or ta

In [35]:
def mongo_search(
    query,
    k=5
):

    query_vector = model.encode(

        [query],

        normalize_embeddings=True

    )[0].tolist()


    pipeline = [

        {
            "$vectorSearch": {

                "index":
                    VECTOR_INDEX_NAME,

                "path":
                    "values",

                "queryVector":
                    query_vector,

                "numCandidates":
                    max(
                        100,
                        k * 20
                    ),

                "limit":
                    k,

                "filter": {
                    "metadata.law":
                        "BNSS"
                }
            }
        },

        {
            "$project": {

                "_id": 0,

                "id": 1,

                "metadata": 1,

                "score": {
                    "$meta":
                        "vectorSearchScore"
                }
            }
        }
    ]


    return list(
        collection.aggregate(
            pipeline
        )
    )

In [36]:
results = mongo_search(

    "When can police arrest "
    "a person without a warrant?",

    k=5
)


print(
    "Retrieved:",
    len(results)
)

Retrieved: 5


In [37]:
for result in results:

    print(
        "\nLAW:",
        result[
            "metadata"
        ]["law"]
    )

    print(
        "SECTION:",
        result[
            "metadata"
        ]["section"]
    )

    print(
        "SCORE:",
        round(
            result["score"],
            4
        )
    )

    print(
        result[
            "metadata"
        ]["text"][:500]
    )

    print(
        "=" * 80
    )


LAW: BNSS
SECTION: 58
SCORE: 0.8592
58. No police officer shall detain in custody a person arrested without warrant for a
longer period than under all the circumstances of the case is reasonable, and such period
shall not, in the absence of a special order of a Magistrate under section 187, exceed
twenty-four hours exclusive of the time necessary for the journey from the place of arrest to
the Magistrate's Court, whether having jurisdiction or not.

LAW: BNSS
SECTION: 57
SCORE: 0.8411
57. A police officer making an arrest without warrant shall, without unnecessary
delay and subject to the provisions herein contained as to bail, take or send the person
arrested before a Magistrate having jurisdiction in the case, or before the officer in charge
of a police station.

LAW: BNSS
SECTION: 78
SCORE: 0.8278
78. The police officer or other person executing a warrant of arrest shall (subject to
the provisions of section 73 as to security) without unnecessary delay bring the person
arrested bef

In [38]:
def safe_mongo_search(
    query,
    k=5
):

    results = mongo_search(
        query,
        k
    )

    if len(results) == 0:

        raise RuntimeError(
            "MongoDB returned 0 BNSS chunks. "
            "Do not call the LLM. "
            "Check collection and test_bns index."
        )

    return results

In [39]:
def extract_section_number(
    query
):

    match = re.search(

        r"\b(?:BNSS\s+)?section\s+"
        r"(\d{1,3})\b",

        query,

        re.IGNORECASE
    )

    if match:

        return (
            match.group(1)
        )

    return None

In [40]:
def retrieve_bnss(
    query,
    k=5
):

    section_number = (
        extract_section_number(
            query
        )
    )


    # -------------------------
    # Exact section lookup
    # -------------------------

    if section_number:

        results = list(
            collection.find(

                {
                    "metadata.law":
                        "BNSS",

                    "metadata.section":
                        section_number
                },

                {
                    "_id": 0
                }
            )
        )


        if not results:

            raise RuntimeError(
                f"BNSS Section "
                f"{section_number} "
                "not found."
            )


        return results


    # -------------------------
    # Semantic vector search
    # -------------------------

    return safe_mongo_search(
        query,
        k
    )

In [41]:
from langchain_groq import ChatGroq

llm = ChatGroq(

    groq_api_key=
        os.getenv(
            "GROQ_API_KEY"
        ),

    model_name=
        "openai/gpt-oss-120b",

    temperature=0
)

In [42]:
def answer_bnss(
    question,
    k=5
):

    results = retrieve_bnss(
        question,
        k=k
    )


    context_parts = []


    for result in results:

        metadata = (
            result["metadata"]
        )


        context_parts.append(

            f"""
Bharatiya Nagarik Suraksha Sanhita, 2023
BNSS Section {metadata['section']}

{metadata['text']}
""".strip()
        )


    context = (
        "\n\n---\n\n".join(
            context_parts
        )
    )


    if not context.strip():

        raise RuntimeError(
            "Context is empty. "
            "LLM call cancelled."
        )


    print(
        "RETRIEVED SECTIONS:"
    )

    for result in results:

        print(
            "BNSS Section",
            result[
                "metadata"
            ]["section"]
        )


    prompt = f"""
You are a legal information assistant.

Answer using ONLY the provided provisions
from the Bharatiya Nagarik Suraksha Sanhita,
2023.

Rules:

1. Mention the relevant BNSS section number.

2. Do not invent provisions, procedures,
conditions or exceptions.

3. Use only the supplied legal context.

4. If none of the supplied provisions contains
the answer, say:
"Insufficient information in retrieved context."


CONTEXT:

{context}


QUESTION:

{question}


ANSWER:
"""


    response = llm.invoke(
        prompt
    )


    print(
        "\nANSWER:\n"
    )

    print(
        response.content
    )


    return response.content

In [43]:
answer_bnss(
    "When can police arrest "
    "a person without a warrant?"
)

RETRIEVED SECTIONS:
BNSS Section 58
BNSS Section 57
BNSS Section 78
BNSS Section 55
BNSS Section 47

ANSWER:

**Police may arrest a person without a warrant when:**

- **A superior officer authorises the arrest in writing.**  
  *BNSS Section 55(1)* requires that an officer in charge or an investigating officer give a subordinate a written order specifying the person to be arrested and the offence or other cause; the subordinate must notify the person of the order (and show it if requested) before making the arrest.

- **The arrest is otherwise permitted by law.**  
  *BNSS Section 55(2)* clarifies that the power to arrest without warrant also exists under other statutory provisions (e.g., Section 35), even without a written order.

- **The arrest is made without unnecessary delay and the arrested person is promptly presented before a magistrate.**  
  *BNSS Section 57* obliges the arresting officer to, without unnecessary delay, bring the person before a magistrate having jurisdiction

'**Police may arrest a person without a warrant when:**\n\n- **A superior officer authorises the arrest in writing.**  \n  *BNSS\u202fSection\u202f55(1)* requires that an officer in charge or an investigating officer give a subordinate a written order specifying the person to be arrested and the offence or other cause; the subordinate must notify the person of the order (and show it if requested) before making the arrest.\n\n- **The arrest is otherwise permitted by law.**  \n  *BNSS\u202fSection\u202f55(2)* clarifies that the power to arrest without warrant also exists under other statutory provisions (e.g., Section\u202f35), even without a written order.\n\n- **The arrest is made without unnecessary delay and the arrested person is promptly presented before a magistrate.**  \n  *BNSS\u202fSection\u202f57* obliges the arresting officer to, without unnecessary delay, bring the person before a magistrate having jurisdiction (or the officer in charge of a police station).\n\n- **The arres